# Modelos de Regressão: OLS e GLM
**Disciplina:** Modelos de Regressão  
**Objetivo:** Entender as limitações da Regressão Linear Clássica e como os GLMs resolvem esses problemas em aplicações de IA.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import boxcox
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# semente aleatoria
np.random.seed(42)
# estilo visual dos gráficos 
sns.set_theme(style="whitegrid")

## 1. Linearidade nos Parâmetros vs. Linearidade nas Variáveis

Um erro conceitual comum é acreditar que a Regressão Linear exige uma relação **linear entre as variáveis**.

Na realidade, o requisito fundamental é a **linearidade nos parâmetros**.

### Exemplo:

Y = bo + b1x² 

Apesar de o modelo ser **não linear em \(x\)**, ele continua sendo **linear nos parâmetros** \(\beta\).

### Análise do Número de Reclamações de Clientes

Carregando dados sintéticos de reclamações de clientes

Características do dataset:
- Variável resposta: Contagem 
- Alta frequência de zeros (muitos clientes sem reclamações)
- Valores extremos (clientes muito insatisfeitos)

In [ ]:
df = pd.read_csv('dataset_reclamacoes.csv', sep=',')

In [ ]:
df

## 4. Análise Exploratória de Dados (EDA)

Vamos investigar:
- Assimetria da variável resposta
- Relação entre média e variância


In [ ]:
df.describe()

In [ ]:
print(f"\nProporção de zeros: {(df['reclamacoes'] == 0).sum() / len(df) * 100:.1f}%")

In [ ]:
# Matriz de gráficos (Pair Plot)
matriz_grafica = sns.pairplot(df, 
                              diag_kind='hist', #hist na diagonal
                              kind='scatter', #correlacao
                              plot_kws={'alpha':0.5, 'color':'teal'},
                              diag_kws={'color':'orange'})
matriz_grafica.fig.suptitle("Análise descritiva: Distribuição e Correlação", y=1.02)
plt.show()

### AJUSTAR MODELO OLS EM DADOS DE CONTAGEM

## Pressupostos do Método dos Mínimos Quadrados (OLS)

O modelo OLS assume:

1. Linearidade nos parâmetros  
2. Independência dos erros  
3. **Homoscedasticidade**  
4. **Normalidade dos resíduos**  
5. Variável resposta contínua  

### Por que OLS falha para dados de contagem?

- Dados de contagem são **discretos**
- Apresentam **assimetria à direita**
- A variância cresce com a média

Exemplos:
- Número de reclamações
- Cliques em anúncios
- Falhas em sistemas
- Casos de doença

---

### Dica
> Quando a variância depende da média, o OLS produz estimativas ineficientes e inferência estatística inválida.


In [ ]:
# Ajustar modelo OLS usando sklearn
X_matriz = df[['tempo_contrato']].values
y = df['reclamacoes'].values

modelo_ols = LinearRegression()
modelo_ols.fit(X_matriz, y)

beta_0_ols = modelo_ols.intercept_
beta_1_ols = modelo_ols.coef_[0]

print("\n📈 Modelo OLS ajustado:")
print(f"\n   Intercepto (β₀): {beta_0_ols:.4f}")
print(f"   Coeficiente (β₁): {beta_1_ols:.4f}")
print(f"   R²: {modelo_ols.score(X_matriz, y):.4f}")

In [ ]:
# Fazer predições
df['pred_ols'] = modelo_ols.predict(X_matriz)

# Identificar predições negativas
pred_negativas = df[df['pred_ols'] < 0]
print(f"\n PROBLEMA CRÍTICO: {len(pred_negativas)} predições NEGATIVAS!")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Gráfico 1: Dispersão com linha de regressão OLS
ax1 = axes[0, 0]
ax1.scatter(df['tempo_contrato'], df['reclamacoes'], alpha=0.5, s=50, 
            label='Dados Reais', color='steelblue')

# Ordenar para plotar linha
sort_idx = np.argsort(df['tempo_contrato'])
ax1.plot(df['tempo_contrato'].iloc[sort_idx], df['pred_ols'].iloc[sort_idx], 
         color='red', linewidth=2, label='Predição OLS', linestyle='--')
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.3)
ax1.set_xlabel('Tempo de Contrato (meses)', fontsize=12)
ax1.set_ylabel('Número de Reclamações', fontsize=12)
ax1.set_title('OLS: A Linha Cruza Zero! ', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Destacar predições negativas
if len(pred_negativas) > 0:
    ax1.scatter(pred_negativas['tempo_contrato'], pred_negativas['pred_ols'], 
                color='red', s=100, marker='x', linewidth=3, 
                label='Predições Negativas', zorder=5)

# Gráfico 2: Resíduos vs Valores Ajustados
residuos_ols = df['reclamacoes'] - df['pred_ols']
ax2 = axes[0, 1]
ax2.scatter(df['pred_ols'], residuos_ols, alpha=0.5, s=50, color='coral')
ax2.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax2.set_xlabel('Valores Ajustados', fontsize=12)
ax2.set_ylabel('Resíduos', fontsize=12)
ax2.set_title('Heterocedasticidade Detectada!' , fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Adicionar linha de tendência
z = np.polyfit(df['pred_ols'], residuos_ols, 2)
p = np.poly1d(z)
x_sorted = np.sort(df['pred_ols'])
ax2.plot(x_sorted, p(x_sorted), "r-", linewidth=2, alpha=0.7, 
         label='Tendência (não-aleatória)')
ax2.legend()

# Gráfico 3: QQ-Plot dos Resíduos
ax3 = axes[1, 0]
stats.probplot(residuos_ols, dist="norm", plot=ax3)
ax3.set_title('QQ-Plot: Resíduos Não-Normais! ', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Gráfico 4: Histograma dos Resíduos
ax4 = axes[1, 1]
ax4.hist(residuos_ols, bins=30, color='lightcoral', edgecolor='black', alpha=0.7)
ax4.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax4.set_xlabel('Resíduos', fontsize=12)
ax4.set_ylabel('Frequência', fontsize=12)
ax4.set_title('Distribuição Assimétrica dos Resíduos', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Problemas do OLS:
- Heterocedasticidade evidente
- Resíduos não normais
- Predições negativas possíveis


### TRANSFORMAÇÃO BOX-COX

In [ ]:
# Box-Cox requer valores positivos
y_boxcox, lambda_optimal = boxcox(df['reclamacoes'] + 1)

print(f"\n Lambda ótimo encontrado: {lambda_optimal:.4f}")
print(f"   (λ = 0 seria log, λ = 1 seria sem transformação)")



In [ ]:
# Ajustar OLS com y transformado
modelo_boxcox = LinearRegression()
modelo_boxcox.fit(X_matriz, y_boxcox)

print("\n Modelo OLS com Box-Cox ajustado:")
print(f"   Intercepto: {modelo_boxcox.intercept_:.4f}")
print(f"   Coeficiente: {modelo_boxcox.coef_[0]:.4f}")
print(f"   R²: {modelo_boxcox.score(X_matriz, y_boxcox):.4f}")

In [ ]:
# Predições na escala transformada
pred_boxcox_transformed = modelo_boxcox.predict(X_matriz)

# Back-transformation
if lambda_optimal != 0:
    pred_boxcox_original = (lambda_optimal * pred_boxcox_transformed + 1)**(1/lambda_optimal) - 1
else:
    pred_boxcox_original = np.exp(pred_boxcox_transformed) - 1

df['pred_boxcox'] = pred_boxcox_original

print("\n  O DILEMA DA INTERPRETAÇÃO:")
print(f"   Coeficiente β₁ = {modelo_boxcox.coef_[0]:.4f}")
print(f"   Mas isso se refere a Y^{lambda_optimal:.4f}, não a Y!")

### Comparar resíduos

In [ ]:
# Comparar resíduos
residuos_boxcox_transformed = y_boxcox - pred_boxcox_transformed
df_plot = df.reset_index().rename(columns={'index': 'id_observacao'})
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

axes[0, 0].scatter(df['pred_ols'], residuos_ols, alpha=0.5, color='coral')
axes[0, 0].axhline(y=0, color='black', linestyle='--')
axes[0, 0].set_xlabel('Valores Ajustados', fontsize=12)
axes[0, 0].set_ylabel('Resíduos', fontsize=12)
axes[0, 0].set_title('OLS Original: Padrão Claro nos Resíduos', fontsize=13, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].scatter(pred_boxcox_transformed, residuos_boxcox_transformed, alpha=0.5, color='mediumseagreen')
axes[0, 1].axhline(y=0, color='black', linestyle='--')
axes[0, 1].set_xlabel('Valores Ajustados (Escala Transformada)', fontsize=12)
axes[0, 1].set_ylabel('Resíduos', fontsize=12)
axes[0, 1].set_title('Box-Cox: Resíduos Melhores, Mas Escala Estranha', fontsize=13, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Gráfico 1: Resíduos OLS por ID 
# Usamos df.index para representar o ID da observação
axes[1, 0].scatter(df.index, residuos_ols, alpha=0.5, color='coral', s=10)
axes[1, 0].axhline(y=0, color='black', linestyle='--')
axes[1, 0].set_xlabel('ID da Observação (Índice)', fontsize=12)
axes[1, 0].set_ylabel('Resíduos', fontsize=12)
axes[1, 0].set_title('OLS: Resíduos por Ordem de ID', fontsize=13, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Gráfico 4: Resíduos Box-Cox por ID 
axes[1, 1].scatter(df.index, residuos_boxcox_transformed, alpha=0.5, color='mediumseagreen', s=10)
axes[1, 1].axhline(y=0, color='black', linestyle='--')
axes[1, 1].set_xlabel('ID da Observação (Índice)', fontsize=12)
axes[1, 1].set_ylabel('Resíduos', fontsize=12)
axes[1, 1].set_title('Box-Cox: Resíduos por Ordem de ID', fontsize=13, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Modelos Lineares Generalizados (GLM)

Um GLM possui três componentes:

1. **Componente Aleatório**  
   Distribuição da família exponencial (Poisson)

2. **Componente Sistemático**

   n = XB

4. **Função de Ligação**

   g(mi)

Para contagens:
- Distribuição: Poisson
- Link: log


### GLM (POISSON)

In [ ]:
def negative_log_likelihood_poisson(params, X, y):
    """Função de verossimilhança negativa para regressão Poisson"""
    beta_0, beta_1 = params
    
    # Predições no espaço log (link canônico)
    log_mu = beta_0 + beta_1 * X
    mu = np.exp(log_mu)
    
    # Log-verossimilhança Poisson: sum[y*log(mu) - mu - log(y!)]
    
    nll = -np.sum(y * log_mu - mu)
    
    return nll

# Valores iniciais (perto dos valores reais)
initial_params = [0.0, 0.0]

# Otimização
resultado = minimize(
    negative_log_likelihood_poisson, 
    initial_params, 
    args=(df['tempo_contrato'].values, df['reclamacoes'].values),
    method='BFGS'
)

beta_0_glm, beta_1_glm = resultado.x

print("\n Modelo GLM (Poisson) ajustado:")
print(f"   Intercepto (β₀): {beta_0_glm:.4f}")
print(f"   Coeficiente (β₁): {beta_1_glm:.4f}")

In [ ]:
# Fazer predições
log_mu_glm = beta_0_glm + beta_1_glm * df['tempo_contrato']
df['pred_glm'] = np.exp(log_mu_glm)

print("\n VANTAGENS DO GLM:")
print("   1. Predições sempre não-negativas (respeitam a natureza dos dados)")
print("   2. Interpretação direta: E[Y|X] sem transformações")
print("   3. Variância modelada corretamente (Var[Y] = E[Y] para Poisson)")
print("   4. Coeficientes interpretáveis na escala log:")
print(f"      β₁ = {beta_1_glm:.4f}")
print(f"      Interpretação: 1 mês a mais → multiplicador de {np.exp(beta_1_glm):.4f}x nas reclamações")

### POR QUE GLM VENCE EM PRODUÇÃO

In [ ]:
# Amostra para comparação
sample_comparison = df.sample(10, random_state=42).sort_values('tempo_contrato')

comparacao = pd.DataFrame({
    'Tempo_Contrato': sample_comparison['tempo_contrato'],
    'Y_Real': sample_comparison['reclamacoes'],
    'Pred_OLS': sample_comparison['pred_ols'],
    'Pred_BoxCox': sample_comparison['pred_boxcox'],
    'Pred_GLM': sample_comparison['pred_glm']
})

print("\n📊 COMPARAÇÃO DE PREDIÇÕES:\n")
print(comparacao.round(2).to_string(index=False))



In [ ]:
# Métricas
mae_ols = mean_absolute_error(df['reclamacoes'], df['pred_ols'])
mae_boxcox = mean_absolute_error(df['reclamacoes'], df['pred_boxcox'])
mae_glm = mean_absolute_error(df['reclamacoes'], df['pred_glm'])

rmse_ols = np.sqrt(mean_squared_error(df['reclamacoes'], df['pred_ols']))
rmse_boxcox = np.sqrt(mean_squared_error(df['reclamacoes'], df['pred_boxcox']))
rmse_glm = np.sqrt(mean_squared_error(df['reclamacoes'], df['pred_glm']))

print("\n📈 MÉTRICAS DE PERFORMANCE:")
print(f"\n{'Modelo':<12} {'MAE':<10} {'RMSE':<10}")
print("-" * 32)
print(f"{'OLS':<12} {mae_ols:<10.3f} {rmse_ols:<10.3f}")
print(f"{'Box-Cox':<12} {mae_boxcox:<10.3f} {rmse_boxcox:<10.3f}")
print(f"{'GLM':<12} {mae_glm:<10.3f} {rmse_glm:<10.3f} ")

In [ ]:
# Visualização final
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sort_idx = np.argsort(df['tempo_contrato'])

# OLS
ax1 = axes[0]
ax1.scatter(df['tempo_contrato'], df['reclamacoes'], alpha=0.4, s=30, 
            color='gray', label='Dados Reais')
ax1.plot(df['tempo_contrato'].iloc[sort_idx], df['pred_ols'].iloc[sort_idx], 
         'r--', linewidth=2, label='OLS', alpha=0.8)
ax1.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.3)
ax1.set_xlabel('Tempo de Contrato (meses)', fontsize=11)
ax1.set_ylabel('Reclamações', fontsize=11)
ax1.set_title('OLS: Predições Negativas! ❌', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(bottom=-5)

# Box-Cox
ax2 = axes[1]
ax2.scatter(df['tempo_contrato'], df['reclamacoes'], alpha=0.4, s=30, 
            color='gray', label='Dados Reais')
ax2.plot(df['tempo_contrato'].iloc[sort_idx], df['pred_boxcox'].iloc[sort_idx], 
         'orange', linewidth=2, label='Box-Cox', alpha=0.8)
ax2.set_xlabel('Tempo de Contrato (meses)', fontsize=11)
ax2.set_ylabel('Reclamações', fontsize=11)
ax2.set_title('Box-Cox: Escala Estranha ⚠️', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# GLM
ax3 = axes[2]
ax3.scatter(df['tempo_contrato'], df['reclamacoes'], alpha=0.4, s=30, 
            color='gray', label='Dados Reais')
ax3.plot(df['tempo_contrato'].iloc[sort_idx], df['pred_glm'].iloc[sort_idx], 
         'green', linewidth=2, label='GLM (Poisson)', alpha=0.8)
ax3.set_xlabel('Tempo de Contrato (meses)', fontsize=11)
ax3.set_ylabel('Reclamações', fontsize=11)
ax3.set_title('GLM: Escala Original! ✅', fontsize=13, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()